In [ ]:
%pip install pymilvus

In [3]:
import os
from dotenv import load_dotenv
# from pymilvus import connections

# # If using Docker standalone Milvus
# connections.connect("default", host="127.0.0.1", port="19530")

from pymilvus import connections

load_dotenv(override=True, dotenv_path="../.env.local")

milvus_uri = os.getenv("MILVUS_URI")
milvus_token = os.getenv("MILVUS_API_KEY")


connections.connect(
    alias="default",
    uri=milvus_uri,
    token=milvus_token
)

print("Connected to Milvus on Zilliz Cloud")

/var/folders/nd/jb8gtjc91h96624zw01gt30w0000gn/T/ipykernel_70122/3933387743.py:16: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(


Connected to Milvus on Zilliz Cloud


In [4]:
from pymilvus import db
from pymilvus import Collection, FieldSchema, CollectionSchema, DataType

# 1. Create a new database
db.create_database("rag_db")

# 2. Switch to that database
db.using_database("rag_db")


/var/folders/nd/jb8gtjc91h96624zw01gt30w0000gn/T/ipykernel_70122/4198949007.py:5: PyMilvusDeprecationWarning: `db.create_database` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  db.create_database("rag_db")
2026-06-15 19:48:32,598 [ERROR][_log_rpc_error]: grpc RpcError: [create_database], <_InactiveRpcError: StatusCode.PERMISSION_DENIED, PrivilegeCreateDatabase: permission deny to db_877c25cd1da2bef in the `db_877c25cd1da2bef` database>, <elapsed:686.5ms>
Traceback:
Traceback (most recent call last):
  File "/Users/suneelsharma/Documents/sweta/sweta-ai-labs/.venv/lib/python3.14/site-packages/pymilvus/decorators.py", line 518, in handler
    return func(*args, **kwargs)
  File "/Users/suneelsharma/Documents/sweta/sweta-ai-labs/.venv/lib/python3.14/site-packages/pymilvus/decorators.py", line 565, in handler
    return func(self, *args, **kwargs)
  File "/Users/suneelsharma/Documents/sweta/sweta-ai-labs/.venv/lib/python3.14/site-packages/py

_InactiveRpcError: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.PERMISSION_DENIED
	details = "PrivilegeCreateDatabase: permission deny to db_877c25cd1da2bef in the `db_877c25cd1da2bef` database"
	debug_error_string = "UNKNOWN:Error received from peer  {grpc_status:7, grpc_message:"PrivilegeCreateDatabase: permission deny to db_877c25cd1da2bef in the `db_877c25cd1da2bef` database"}"
>

In [6]:

# ----- Create schema -----
fields = [
    FieldSchema("doc_id", DataType.INT64, is_primary=True, auto_id=False),
    FieldSchema("title", DataType.VARCHAR, max_length=200),
    FieldSchema("domain", DataType.VARCHAR, max_length=100),
    FieldSchema("content", DataType.VARCHAR, max_length=2000),
    FieldSchema("embedding", DataType.FLOAT_VECTOR, dim=384) 
]

schema = CollectionSchema(fields, description="Policy documents with embeddings")
collection = Collection("policy_docs_collection", schema)

# ----- Create index -----
index_params = {
    "index_type": "IVF_FLAT",
    "metric_type": "COSINE",
    "params": {"nlist": 128},
}
collection.create_index(field_name="embedding", index_params=index_params)

/var/folders/nd/jb8gtjc91h96624zw01gt30w0000gn/T/ipykernel_70122/625364760.py:11: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection = Collection("policy_docs_collection", schema)
/var/folders/nd/jb8gtjc91h96624zw01gt30w0000gn/T/ipykernel_70122/625364760.py:19: PyMilvusDeprecationWarning: `Collection.create_index` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.create_index(field_name="embedding", index_params=index_params)


Status(code=0, message=)

In [7]:
# ----- Example data -----
content_chunks = [
    {
        "doc_id": 1,
        "section": "Pay Policies",
        "title": "Employee Pay Policy",
        "domain": "Human Resources",
        "content": "Employees are paid bi-weekly via direct deposit."
    },
    {
        "doc_id": 1,
        "section": "Leave of Absence",
        "title": "Leave Request and Approval Process",
        "domain": "Human Resources",
        "content": "Employees must submit a leave request for approval."
    },
    {
        "doc_id": 1,
        "section": "Internet Use",
        "title": "Acceptable Use of Company Internet",
        "domain": "IT & Security",
        "content": "Company internet must be used for work-related tasks only."
    },
    {
        "doc_id": 2,
        "section": "Break at Work",
        "title": "Employee Break Policy",
        "domain": "Workplace Operations",
        "content": "Employees can take an hour break."
    },
    {
        "doc_id": 2,
        "section": "Harassment",
        "title": "Workplace Harassment Policy",
        "domain": "Compliance",
        "content": "Interact with each employee with respect."
    }
]


# content_chunks_list = []
# for chunk in content_chunks:
#     content_chunks_list.append(chunk["content"])
content_chunks_list = [chunk["content"] for chunk in content_chunks]
print(content_chunks_list)

['Employees are paid bi-weekly via direct deposit.', 'Employees must submit a leave request for approval.', 'Company internet must be used for work-related tasks only.', 'Employees can take an hour break.', 'Interact with each employee with respect.']


In [8]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")

doc_vectors = model.encode(content_chunks_list)
doc_vectors.shape

/Users/suneelsharma/Documents/sweta/sweta-ai-labs/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7943.32it/s]


(5, 384)

In [9]:
# ---- Build columnar data ----
doc_ids = [int(i + 1) for i in range(len(content_chunks))]             # INT64
titles = [str(doc["title"]) for doc in content_chunks]                 # VARCHAR
domains = [str(doc["domain"]) for doc in content_chunks]               # VARCHAR
content = [str(doc["content"]) for doc in content_chunks]               # VARCHAR
embeddings = [list(map(float, vec)) for vec in doc_vectors]       # FLOAT_VECTOR(768)


# ---- Insert column-wise ----
collection.insert([doc_ids, titles, domains, content, embeddings])
collection.flush()

print(f"Successfully inserted {len(doc_ids)} documents into Milvus.")

/var/folders/nd/jb8gtjc91h96624zw01gt30w0000gn/T/ipykernel_70122/50992026.py:10: PyMilvusDeprecationWarning: `Collection.insert` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.insert([doc_ids, titles, domains, content, embeddings])
/var/folders/nd/jb8gtjc91h96624zw01gt30w0000gn/T/ipykernel_70122/50992026.py:11: PyMilvusDeprecationWarning: `Collection.flush` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.flush()


Successfully inserted 5 documents into Milvus.


In [10]:
#Load the collection before searching or querying
collection.load()
res = collection.query(expr="doc_id > 0", 
        output_fields=["doc_id", "title", "domain", "content", "embedding"])
print(res)

/var/folders/nd/jb8gtjc91h96624zw01gt30w0000gn/T/ipykernel_70122/4085184805.py:2: PyMilvusDeprecationWarning: `Collection.load` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.load()
/var/folders/nd/jb8gtjc91h96624zw01gt30w0000gn/T/ipykernel_70122/4085184805.py:3: PyMilvusDeprecationWarning: `Collection.query` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  res = collection.query(expr="doc_id > 0",


data: ["{'doc_id': 1, 'title': 'Employee Pay Policy', 'domain': 'Human Resources', 'content': 'Employees are paid bi-weekly via direct deposit.', 'embedding': [0.024725139141082764, -0.009081494994461536, 0.03887123614549637, 0.03230157122015953, -0.058905504643917084, 0.039124589413404465, 0.02601798251271248, -0.04369088262319565, -0.019160054624080658, -1.1309633691780618e-06, 0.013653445057570934, 0.004326747730374336, -0.05378558859229088, 0.0068689011968672276, 0.0076382881961762905, -0.06069638952612877, -0.03760223090648651, 0.007140920963138342, 0.13387173414230347, 0.001735408091917634, -0.0030755840707570314, -0.07781299203634262, -0.08451437205076218, -0.003216363489627838, 0.13382810354232788, -0.03643584996461868, 0.01692274585366249, 0.04861479252576828, -0.04028795287013054, 0.019676703959703445, -0.04371286928653717, 0.05151142552495003, -0.016299612820148468, -0.03905921056866646, -0.0044538602232933044, 0.00047449758858419955, -0.031205127015709877, 0.048956222832202

In [11]:
# print(utility.has_collection("demo_collection"))

# # Get details about a specific collection
# # Get collection details
# collection = Collection("demo_collection")  # instantiate the collection object
print(collection.schema)                    # show the schema
print(collection.num_entities)              # number of entities
print(collection.description)               # optional

{'auto_id': False, 'description': 'Policy documents with embeddings', 'fields': [{'name': 'doc_id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': False}, {'name': 'title', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 200}}, {'name': 'domain', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 100}}, {'name': 'content', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 2000}}, {'name': 'embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 384}}], 'enable_dynamic_field': False, 'enable_namespace': False}
5
Policy documents with embeddings


/var/folders/nd/jb8gtjc91h96624zw01gt30w0000gn/T/ipykernel_70122/2025431850.py:7: PyMilvusDeprecationWarning: `Collection.num_entities` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  print(collection.num_entities)              # number of entities


In [12]:
query = "What’s the leave policy?"
query_vector = model.encode([query])[0]
query_vector[:5]  # Show only first 5 values

array([0.06045819, 0.03882856, 0.01172255, 0.03124588, 0.12124222],
      dtype=float32)

In [13]:
# Search for closest match only in the 'Human Resources' domain
results = collection.search(
    data=[query_vector],
    anns_field="embedding",
    param={"metric_type": "COSINE", "params": {"nprobe": 10}},
    limit=3,
    # expr='domain == "Human Resources"',
    output_fields=["doc_id", "title", "domain", "content"]
)

context_sring = ""
for res in results[0]:
    print(f"doc_id={res.entity.get('doc_id')}, "
          f"title={res.entity.get('title')}, "
          f"domain={res.entity.get('domain')}, "
          f"content={res.entity.get('content')}, "
          f"score={res.distance}")
    context_sring += f"\n -- \n {res.entity.get('content')} " # Append content to context string

print("\nContext String for RAG:\n", context_sring) 

/var/folders/nd/jb8gtjc91h96624zw01gt30w0000gn/T/ipykernel_70122/2492072602.py:2: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results = collection.search(


doc_id=2, title=Leave Request and Approval Process, domain=Human Resources, content=Employees must submit a leave request for approval., score=0.6198837161064148
doc_id=4, title=Employee Break Policy, domain=Workplace Operations, content=Employees can take an hour break., score=0.3879220187664032
doc_id=1, title=Employee Pay Policy, domain=Human Resources, content=Employees are paid bi-weekly via direct deposit., score=0.23205098509788513

Context String for RAG:
 
 -- 
 Employees must submit a leave request for approval. 
 -- 
 Employees can take an hour break. 
 -- 
 Employees are paid bi-weekly via direct deposit. 


In [ ]:
from llm_utlity import ask_question_open_ai 

query = "What’s the leave policy?"
response = ask_question_open_ai(query, context_sring)
response

In [ ]:
print(f"User query: {query}")
print(f"Context: {context_sring}")

print(f"\n\nOpen AI Response: {response}")